# Documento para aprender a biblioteca PyTorch

In [2]:
import torch
from torch import nn
import numpy as np
import pandas as pd

## 1 - Modelo extremamente simples, apenas para apreender a sintaxe do PyTorch

In [3]:
class ModeloSimples(nn.Module):
    def __init__(self):
        super().__init__();
        self.linear1 = nn.Linear(3, 4);
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(4, 1);
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x);
        x = self.linear2(x);
        return x;

In [4]:
modelo = ModeloSimples()

x = torch.tensor([
    [1.0,2.0,3.0],
    [4.0,5.0,6.0]
])

saida = modelo(x)

print(saida)

tensor([[-0.7183],
        [-1.6977]], grad_fn=<AddmmBackward0>)


## 2 - Modelo com um treinamento simples, loss MSE e otimizador Adam

In [5]:
class ModeloComTreino(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(5, 3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3, 1)
    
    def forward(self, x):
        x = self.linear1(x);
        x = self.relu(x);
        x = self.linear2(x);
        return x;
        

In [6]:
modelo = ModeloComTreino()

x = torch.tensor([
    [1.0, 2.0, 3.0, 7.0, 3.0],
    [4.0, 9.0, 33.0, 6.0, 1.0],
    [7.0, 2.0, 22.0, 1.0, 6.0],
])

y = torch.tensor([
  [1.0],
  [1.0],
  [1.0]
])

criterio = nn.MSELoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

# loop de treinamento
for epoch in range(250):
  # forward
  predicao = modelo(x)
  
  # calcular loss
  loss = criterio(predicao, y)
  
  # resetar gradientes
  otimizador.zero_grad()
  
  # backpropagation
  loss.backward()
  
  # atualizar pesos
  otimizador.step()
  
  print(f"Epoch {epoch} -> Loss: {loss.item()}")


with torch.no_grad():
  saida = modelo(x)
  
print(saida)

Epoch 0 -> Loss: 0.5524072647094727
Epoch 1 -> Loss: 0.33611440658569336
Epoch 2 -> Loss: 0.21083401143550873
Epoch 3 -> Loss: 0.15653224289417267
Epoch 4 -> Loss: 0.14756174385547638
Epoch 5 -> Loss: 0.1599569469690323
Epoch 6 -> Loss: 0.1750209927558899
Epoch 7 -> Loss: 0.1819211095571518
Epoch 8 -> Loss: 0.17700235545635223
Epoch 9 -> Loss: 0.16135601699352264
Epoch 10 -> Loss: 0.13851790130138397
Epoch 11 -> Loss: 0.11285781115293503
Epoch 12 -> Loss: 0.0885453000664711
Epoch 13 -> Loss: 0.06885324418544769
Epoch 14 -> Loss: 0.05566536262631416
Epoch 15 -> Loss: 0.04919539391994476
Epoch 16 -> Loss: 0.048032063990831375
Epoch 17 -> Loss: 0.04961824044585228
Epoch 18 -> Loss: 0.05110365152359009
Epoch 19 -> Loss: 0.050273943692445755
Epoch 20 -> Loss: 0.04616667330265045
Epoch 21 -> Loss: 0.039156246930360794
Epoch 22 -> Loss: 0.03057115338742733
Epoch 23 -> Loss: 0.022086746990680695
Epoch 24 -> Loss: 0.015146027319133282
Epoch 25 -> Loss: 0.010573498904705048
Epoch 26 -> Loss: 0.0

## 3 - Modelo para series temporais (CNN 1D) usando dados ficticios

In [13]:
class CNN1D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=16, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=8, out_channels=1, kernel_size=3, padding=1)
        )
    
    def forward(self, x):
        return self.net(x)

In [19]:
# X
temperatura = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
pressao = [2.0, 2.2, 2.4, 2.1, 2.12, 2.5]

X = torch.tensor([temperatura, pressao], dtype=torch.float32) # (C, L)
X = X.unsqueeze(0) # (1, C, L)

# Y
velocidade_molecula = [0.0, 3.0, 6.0, 8.0, 9.0, 10.0]

Y = torch.tensor([velocidade_molecula], dtype=torch.float32)
Y = Y.unsqueeze(0) # (1, 1, L)


print(X.shape)
print(Y.shape)

torch.Size([1, 2, 6])
torch.Size([1, 1, 6])


In [20]:
modelo = CNN1D(in_channels=2)

criterio = nn.MSELoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01) # o que sao os modelo.parameters()

for epoch in range(100):
    modelo.train()
    
    predicao = modelo(X)
    loss = criterio(predicao, Y)
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    
    #Entender melhor backpropagation e como o otimizador atualiza os pesos
    if(epoch % 10 == 0):
      print(f"Epoch {epoch} -> Loss: {loss.item()}")
    

modelo.eval()
with torch.no_grad():
  saida = modelo(X)
  
print(saida)

Epoch 0 -> Loss: 44.78120422363281
Epoch 10 -> Loss: 6.03819465637207
Epoch 20 -> Loss: 1.8844432830810547
Epoch 30 -> Loss: 0.39792314171791077
Epoch 40 -> Loss: 0.445791095495224
Epoch 50 -> Loss: 0.3060213625431061
Epoch 60 -> Loss: 0.20080877840518951
Epoch 70 -> Loss: 0.12885938584804535
Epoch 80 -> Loss: 0.06335853040218353
Epoch 90 -> Loss: 0.02191544882953167
tensor([[[ 0.0854,  2.9473,  6.0016,  7.9982,  9.1081, 10.0035]]])


## 3 - Modelo Classificador Teste

In [7]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random
import numpy as np

# Reprodutibilidade
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset sintetico
X, Y = make_classification(
  n_samples=2000,
  n_features=10,
  n_informative=6,
  n_redundant=2,
  n_classes=3,
  random_state=seed
)

#Normalizar e remover NaNs
scaler = StandardScaler()
X = scaler.fit_transform(X) # Para classificacao, normalizar apenas o X

X = torch.tensor(X, dtype=torch.float32)
X = torch.nan_to_num(X)

Y = torch.tensor(Y, dtype=torch.long)

# Split para treino e validacao (stratify para garantir que a proporcao de classes seja a mesma)
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=seed, stratify=Y)


train_ds = TensorDataset(X_train, Y_train)
val_ds = TensorDataset(X_val, Y_val)


batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)


class MLP_Classifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(10, 10),
      nn.ReLU(),
      nn.Linear(10, 10),
      nn.ReLU(),
      nn.Linear(10, 3),
    )

  def forward(self, x):
    return self.net(x)


# Funcoes auxiliares
def accuracy(preds, targets):
    preds_class = preds.argmax(dim=1)
    return (preds_class == targets).float().mean().item()

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            acc = accuracy(logits, yb)
            batch_size = xb.size(0)
            total_loss += loss.item() * batch_size
            total_acc += acc * batch_size
            n += batch_size
    return total_loss / n, total_acc / n


# Inicializacao

modelo = MLP_Classifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=1e-3)

# Treinamento
epochs = 10
best_val_acc = 0.0

for epoch in range(epochs):
  modelo.train()
  running_loss = 0.0
  running_acc = 0.0
  n_train = 0

  for xb, yb in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    logits = modelo(xb)
    loss = criterion(logits, yb)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * xb.size(0)
    running_acc += (logits.argmax(dim=1) == yb).float().sum().item()
    n_train += xb.size(0)

  train_loss = running_loss / n_train
  train_acc  = running_acc / n_train

  val_loss, val_acc = evaluate(modelo, val_loader)

  if val_acc > best_val_acc:
          best_val_acc = val_acc
          best_state = modelo.state_dict()

  print(f"Epoch {epoch:2d} | Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | Val loss: {val_loss:.4f}, acc: {val_acc:.4f}")


print(f"Melhor val acc: {best_val_acc:.4f}")
modelo.load_state_dict(best_state)

Epoch  0 | Train loss: 1.1052, acc: 0.3344 | Val loss: 1.0981, acc: 0.3500
Epoch  1 | Train loss: 1.0901, acc: 0.4006 | Val loss: 1.0835, acc: 0.4300
Epoch  2 | Train loss: 1.0751, acc: 0.4838 | Val loss: 1.0658, acc: 0.4850
Epoch  3 | Train loss: 1.0553, acc: 0.5400 | Val loss: 1.0447, acc: 0.5150
Epoch  4 | Train loss: 1.0295, acc: 0.5587 | Val loss: 1.0142, acc: 0.5575
Epoch  5 | Train loss: 0.9943, acc: 0.5844 | Val loss: 0.9794, acc: 0.5900
Epoch  6 | Train loss: 0.9548, acc: 0.6100 | Val loss: 0.9391, acc: 0.6150
Epoch  7 | Train loss: 0.9122, acc: 0.6369 | Val loss: 0.8977, acc: 0.6400
Epoch  8 | Train loss: 0.8676, acc: 0.6625 | Val loss: 0.8582, acc: 0.6450
Epoch  9 | Train loss: 0.8263, acc: 0.6681 | Val loss: 0.8222, acc: 0.6550
Melhor val acc: 0.6550


<All keys matched successfully>

### Prever um exemplo com o modelo classificador

In [ ]:
with torch.no_grad():
  modelo.eval()
  saida = modelo(X_val[0]).to(device)